# proto-REINFORCE, Last-Minute Pricing, part 1: episodes and returns

This morning you sold the seats of a small plane in the arcade: 5 seats, 4 days to takeoff, one price a day,
PREMIUM at \$5 a seat or STANDARD at \$2. proto-REINFORCE learns to price it from episodes of the plane, and here an
episode is a flight. This notebook builds a flight in Python, one small piece at a time: a probability table, a
price sampled with a roulette, a day of the plane, a flight, and its return, the flight's MONEY.
Part 2 builds the forward pass, part 3 the backward pass.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one
only if you are still stuck. After each task, a check cell tells you whether your code works. The luck of the
game is in the kit: your code never deals with dice.

Start with the two folded cells below: the plane's kit and the checks.

In [ ]:
#@title The plane's kit (run me first) { display-mode: "form" }
# Last-Minute Pricing's kit. Run this cell first; you do not need to read it.
import random                   # Python's random numbers: random.randint(0, 14) gives 0, 1, 2, ..., 14
from math import exp            # e to the power: exp(0.4) is 1.4918...
import math as _kit_math

SEATS = 5                       # the seats of the plane
DAYS = 4                        # the days to takeoff: DAY 1, 2, 3 and 4
PRICES = ['PREMIUM', 'STANDARD']
PREMIUM, STANDARD = 0, 1                # the columns of a table: the prices
DOLLARS = [5, 2]                  # a seat costs $5 at PREMIUM and $2 at STANDARD
FLIGHTS = 2500                  # the flights of every average
# The rows of a table: every situation a flight can be in, [seats left, day]. STANDARD sells at most 3 seats a
# day, so on DAY 2 at least 2 seats are left; on DAY 3 and DAY 4 anything from 1 to 5.
SITUATIONS = [[5, 1], [5, 2], [4, 2], [3, 2], [2, 2], [5, 3], [4, 3], [3, 3], [2, 3], [1, 3], [5, 4], [4, 4], [3, 4], [2, 4], [1, 4]]

# The plane of the arcade (levers.js and pricing.js): how many buyers come in a day, with their probabilities
_KIT_BUYERS = [[[0, 0.6], [1, 0.4]], [[1, 0.2], [2, 0.4], [3, 0.4]]]
_KIT_START_PREFERENCES = [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]
_KIT_BEST_LINE = 11.8392       # the optimal policy's average return on the flights of update 240
_KIT_BEST = [0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 2]      # the best price of each row (2: either)
_KIT_LOSES = [0.0256, 0.224, 0.064, 0.96, 2.016, 2.08, 1.12, 0.16, 0.96, 1.2, 2.4, 2.4, 2.4, 1.6, 0.0]      # what the other price loses there, on average
_KIT_OFTEN = [1.0, 0.6, 0.4, 0.0, 0.0, 0.0, 0.36, 0.4, 0.24, 0.0, 0.0, 0.0, 0.312, 0.448, 0.24]      # the share of the optimal policy's flights that are in that row
_KIT_FIRST_LUCK = 12                # the luck of the start
_KIT_LUCK_BASE = 2000              # update i of the loop flies the luck of seed 2000 + i
_KIT_PICKS = 1                       # the seed of Python's random for the loop's picks (play sets it)


def _kit_sell(seats_left, price, u):
    """One day of the plane on a given number u between 0 and 1 (pricing.js's sample): the seats sold and the
    money."""
    if isinstance(seats_left, bool) or seats_left not in (1, 2, 3, 4, 5) or isinstance(price, bool) or \
            price not in (0, 1):
        raise ValueError("sell needs 1 to 5 seats left and a price, 0 for PREMIUM or 1 for STANDARD; it got %r "
                         "seats left and the price %r" % (seats_left, price))
    running = 0
    buyers = _KIT_BUYERS[price][-1][0]
    for count, probability in _KIT_BUYERS[price]:
        running = running + probability      # the running probabilities, added as pricing.js adds them
        if u < running:
            buyers = count
            break
    seats_sold = min(buyers, seats_left)
    return seats_sold, DOLLARS[price] * seats_sold


# The luck: one stream of numbers between 0 and 1 (Mulberry32, the generator of the arcade), two numbers a day
_KIT_LUCK = {"seed": _KIT_FIRST_LUCK, "state": _KIT_FIRST_LUCK, "update": 0, "day": 0}


def _kit_next():
    s = (_KIT_LUCK["state"] + 0x6D2B79F5) & 0xFFFFFFFF
    _KIT_LUCK["state"] = s
    t = ((s ^ (s >> 15)) * (1 | s)) & 0xFFFFFFFF
    t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
    return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296


def spin():
    """The ball of a roulette: a number between 0 and 1, every number as likely as any other."""
    return _kit_next()


def sell(seats_left, price):
    """One day of the plane. With this many seats left and this price, luck decides how many buyers come, with the
    game's probabilities; they buy what is left. Gives the seats sold and the day's money:
        seats_sold, money = sell(5, STANDARD)"""
    if seats_left is Ellipsis or price is Ellipsis:
        print("⏳ sell waits for the seats left and a price: replace each ... with yours, then run the cell again.")
        return None, None
    seats_sold, money = _kit_sell(seats_left, price, _kit_next())
    _KIT_LUCK["day"] = _KIT_LUCK["day"] + 1
    if seats_sold == seats_left or _KIT_LUCK["day"] >= DAYS:
        # the flight is over: the luck of its unused days is skipped, so the next flight starts on its own numbers
        while _KIT_LUCK["day"] < DAYS:
            _kit_next()
            _kit_next()
            _KIT_LUCK["day"] = _KIT_LUCK["day"] + 1
        _KIT_LUCK["day"] = 0
    return seats_sold, money


def same_luck():
    """The luck starts again from its beginning: the flights that follow meet the same buyers as the flights of every
    average before them, so two tables of preferences are compared on the same luck."""
    _KIT_LUCK["state"] = _KIT_LUCK["seed"]
    _KIT_LUCK["day"] = 0


def new_luck():
    """New flights for the next update of proto-REINFORCE: the luck starts from a new seed, 2000 plus the number of
    the update."""
    _KIT_LUCK["update"] = _KIT_LUCK["update"] + 1
    _KIT_LUCK["seed"] = _KIT_LUCK_BASE + _KIT_LUCK["update"]
    _KIT_LUCK["state"] = _KIT_LUCK["seed"]
    _KIT_LUCK["day"] = 0


def _kit_set_luck(seed, update=0):
    """For the checks: the luck of this seed, from its beginning."""
    _KIT_LUCK["seed"] = seed
    _KIT_LUCK["state"] = seed
    _KIT_LUCK["update"] = update
    _KIT_LUCK["day"] = 0


def _kit_before_long():
    """play calls this before the loop: the luck of the start, and the same picks for everyone."""
    _kit_set_luck(_KIT_FIRST_LUCK)
    random.seed(_KIT_PICKS)


def _kit_after_long():
    """play calls this after the loop: the luck of the start again, for the cells above."""
    _kit_set_luck(_KIT_FIRST_LUCK)


def _kit_seed_picks(seed):
    """For the checks: Python's random from this seed."""
    random.seed(seed)


def _kit_save():
    """For the checks: the state of the luck and of Python's random."""
    return [dict(_KIT_LUCK), random.getstate()]


def _kit_restore(saved):
    """For the checks: the luck and Python's random as they were."""
    _KIT_LUCK.update(saved[0])
    random.setstate(saved[1])


def start_preferences():
    """The starting table of preferences, picked at random between -10 and 10: a fresh copy each time."""
    return copy_preferences(_KIT_START_PREFERENCES)


def copy_preferences(preferences):
    """A copy of a table of preferences: change the copy, and the table stays as it was."""
    copy = []
    for row in preferences:
        copy.append(list(row))
    return copy


def _kit_situation(row):
    """DAY 1, 5 SEATS LEFT"""
    seats_left, day = SITUATIONS[row]
    return "DAY %d, %d %s LEFT" % (day, seats_left, "SEAT" if seats_left == 1 else "SEATS")


def show_situations():
    """Print the situations a flight can be in, one per row of a table."""
    print("row   situation             [seats left, day]")
    for row in range(len(SITUATIONS)):
        print("%3d   %-20s  %s" % (row, _kit_situation(row), SITUATIONS[row]))


def _kit_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _kit_is_table(table):
    if not isinstance(table, list) or len(table) != len(SITUATIONS):
        return False
    for row in table:
        if not isinstance(row, list) or len(row) != 2:
            return False
        for x in row:
            if not _kit_is_number(x):
                return False
    return True


def _kit_short(value):
    text = repr(value).replace("Ellipsis", "...")
    return text if len(text) <= 60 else text[:57] + "..."


def _kit_print_table(table, places, mark):
    head = "row   situation           " + "%10s%10s" % (PRICES[0], PRICES[1])
    if mark:
        head += "    likeliest"
    print(head)
    for row in range(len(SITUATIONS)):
        line = "%3d   %-20s" % (row, _kit_situation(row)) + "%10.*f%10.*f" % (places, table[row][0], places,
                                                                            table[row][1])
        if mark:
            line += "    " + PRICES[0 if table[row][0] > table[row][1] else 1]
        print(line)


def show_preferences(preferences):
    """Print a table of preferences: a row per situation, a column per price, two decimals. Silent for None (a line
    above says why)."""
    if preferences is None:
        return
    if not _kit_is_table(preferences):
        print("⏳ No table of preferences to show yet: show_preferences needs %d rows of two numbers. It got %s."
              % (len(SITUATIONS), _kit_short(preferences)))
        return
    _kit_print_table(preferences, 2, False)


def show_probabilities(table):
    """Print a probability table (a policy): a row per situation, a column per price, two decimals, and the
    likeliest price of each row. Silent for None (a line above says why)."""
    if table is None:
        return
    if not _kit_is_table(table):
        print("⏳ No probability table to show yet: show_probabilities needs %d rows of two numbers. It got %s."
              % (len(SITUATIONS), _kit_short(table)))
        return
    _kit_print_table(table, 2, True)


def show_counts(counts):
    """Print a table of counts: a row per situation, a column per price, whole numbers."""
    if counts is None:
        return
    if not _kit_is_table(counts):
        print("⏳ No counts to show yet: show_counts needs %d rows of two numbers. It got %s."
              % (len(SITUATIONS), _kit_short(counts)))
        return
    _kit_print_table(counts, 0, False)


def show_roulette(probabilities, ball=None):
    """Print the roulette of a row of probabilities: each price's slot, starting where the slot before it ends, on a
    wheel of 50 marks from 0 to 1. With a ball, a ^ marks where it is."""
    ok = isinstance(probabilities, list) and len(probabilities) == len(PRICES)
    if ok:
        for x in probabilities:
            if not _kit_is_number(x) or x < 0:
                ok = False
    if not ok:
        print("⏳ No roulette to show yet: show_roulette needs a row of two probabilities. It got %s."
              % _kit_short(probabilities))
        return
    start = 0
    for a in range(len(PRICES)):
        end = start + probabilities[a]
        left = int(round(start * 50))
        right = int(round(end * 50))
        print("%-9s %4.2f to %4.2f  |%s%s" % (PRICES[a], start, end, " " * left, "#" * (right - left)))
        start = end
    print(" " * 25 + "0" + " " * 49 + "1")
    if ball is not None and _kit_is_number(ball) and 0 <= ball < 1:
        print(" " * 25 + " " * int(ball * 50) + "^ the ball, %.2f" % ball)


def _kit_dollars(x):
    """$5, $10.2596: whole dollars without cents, an average with every digit (at most 4 decimals)."""
    if x == int(x):
        text = "$%d" % abs(int(x))
    else:
        text = ("$%.4f" % abs(x)).rstrip("0")
    return "-" + text if x < 0 else text


def show_day(seats_sold, money):
    """Print one day of the plane: the seats sold and the day's money. Silent when there is no day yet (a line above
    says why)."""
    if seats_sold is None and money is None:
        return
    if not _kit_is_number(seats_sold) or not _kit_is_number(money):
        print("⏳ No day to show yet: show_day needs the seats sold and the day's money. It got %s and %s."
              % (_kit_short(seats_sold), _kit_short(money)))
        return
    print("Seats sold: %d. The day's money: %s." % (seats_sold, _kit_dollars(money)))


def show_money(label, money):
    """Print an amount of money, or a list of amounts, after a label: show_money("MONEY:", 12) prints MONEY: $12.
    A long list shows its first 10. Silent when there is nothing to show yet (a line above says why)."""
    if money is None:
        return
    if isinstance(money, list):
        parts = []
        for k in range(len(money)):
            if k == 10:
                parts.append("... (%s in all)" % format(len(money), ","))
                break
            if _kit_is_number(money[k]):
                parts.append(_kit_dollars(money[k]))
            else:
                parts.append(_kit_short(money[k]))
        print(label, ", ".join(parts) if parts else "none")
    elif _kit_is_number(money):
        print(label, _kit_dollars(money))
    else:
        print(label, _kit_short(money))


def show_value(label, value):
    """Print a value after a label: show_value("nabla =", -0.16). Silent for None (a line above says why)."""
    if value is None:
        return
    print(label, _kit_short(value))


def _kit_money(x):
    """$10.18"""
    text = "$%.2f" % abs(x)
    return "-" + text if x < 0 else text


def plot_learning_curve(points, best=_KIT_BEST_LINE, label="optimal policy"):
    """The learning curve, as text: across, the number of updates; up, the average return of a flight after them. o is
    a point (one column for every few updates when there are many), and the row of dashes marks the optimal policy's
    average return on the flights of the last update. Silent for None (a line above says why)."""
    if points is None:
        return
    ok = isinstance(points, list) and len(points) >= 2
    if ok:
        for x in points:
            if not _kit_is_number(x):
                ok = False
    if not ok:
        print("⏳ No learning curve yet: plot_learning_curve needs a list of at least two averages. It got %s."
              % _kit_short(points))
        return
    n = len(points) - 1                      # the updates
    columns = min(n + 1, 61)
    where = []                               # the update that each column shows
    for c in range(columns):
        where.append(int(round(c * n / (columns - 1))))
    values = []
    for i in where:
        values.append(points[i])
    low, high = min(values + [best]), max(values + [best])
    step = 0.1
    for size in (0.1, 0.2, 0.25, 0.5, 1, 2, 5, 10):
        step = size
        if (high - low) / size <= 18:
            break
    top = _kit_math.ceil(round(high / step, 9)) * step
    bottom = _kit_math.floor(round(low / step, 9)) * step
    rows = int(round((top - bottom) / step)) + 1
    grid = []
    for r in range(rows):
        grid.append([" "] * columns)
    best_row = int(round((top - best) / step))
    for c in range(columns):
        grid[best_row][c] = "-"
    for c in range(columns):
        r = int(round((top - values[c]) / step))
        grid[r][c] = "o"
    every = max(1, int(round(n / (columns - 1)))) if columns > 1 else 1
    print("The learning curve: the average return of a flight after each update" +
          (" (one o for every %d updates)" % every if every > 1 else ""))
    print()
    for r in range(rows):
        line = "%8s |" % _kit_money(top - r * step) + "".join(grid[r])
        if r == best_row:
            line += "  %s, %s" % (label, _kit_money(best))
        print(line.rstrip())
    print(" " * 9 + "+" + "-" * columns)
    ticks = 100 if n >= 300 else (50 if n >= 200 else (10 if n >= 40 else 1))
    marks = [" "] * (columns + 6)
    for t in range(0, n + 1, ticks):
        c = int(round(t * (columns - 1) / n)) if n else 0
        text = str(t)
        for j in range(len(text)):
            if c + j < len(marks):
                marks[c + j] = text[j]
    print(" " * 10 + "".join(marks).rstrip() + "   updates")
    print("first point %s, last point %s" % (_kit_money(points[0]), _kit_money(points[-1])))


def show_best(preferences):
    """The optimal policy, row by row, worked out backwards from DAY 4 with the game's probabilities, beside the
    likeliest price of a table of preferences. Silent for None (a line above says why)."""
    if preferences is None:
        return
    if not _kit_is_table(preferences):
        print("⏳ No table of preferences to compare yet: show_best needs %d rows of two numbers. It got %s."
              % (len(SITUATIONS), _kit_short(preferences)))
        return
    print("best:  the best price in this situation, with the best prices after it")
    print("loses: what the other price loses here, on average, in dollars")
    print("often: the share of the optimal policy's flights that are ever in this situation")
    print("yours: the likeliest price of your table")
    print()
    print("row   situation             best       loses   often   yours")
    same, matters = 0, 0
    for row in range(len(SITUATIONS)):
        yours = 0 if preferences[row][0] > preferences[row][1] else 1
        best = "either" if _KIT_BEST[row] == 2 else PRICES[_KIT_BEST[row]]
        mark = ""
        if _KIT_BEST[row] != 2:
            matters += 1
            if yours == _KIT_BEST[row]:
                same += 1
                mark = "  ✓"
            else:
                mark = "  ✗"
        print("%3d   %-20s  %-9s %6s   %4d%%   %-9s%s" % (row, _kit_situation(row), best, _kit_money(_KIT_LOSES[row]),
                                                       round(100 * _KIT_OFTEN[row]), PRICES[yours], mark))
    print()
    print("Your table's likeliest price is the best one in %d of the %d situations where the price matters." % (
        same, matters))


print("✅ Last-Minute Pricing's kit is ready.")

In [ ]:
#@title The checks of part 1 (run me too) { display-mode: "form" }
# The checks of notebook 2 and play. You do not need to read this cell.
# A check prints one short message and never stops the notebook.

_N2_TESTS = {'row_of': [{'about': '5 seats left on DAY 1', 'args': [5, 1], 'kind': 'action', 'want': 0, 'tol': 0}, {'about': '2 seats left on DAY 2', 'args': [2, 2], 'kind': 'action', 'want': 4, 'tol': 0}, {'about': '4 seats left on DAY 3', 'args': [4, 3], 'kind': 'action', 'want': 6, 'tol': 0}, {'about': '3 seats left on DAY 4', 'args': [3, 4], 'kind': 'action', 'want': 12, 'tol': 0}, {'about': '1 seats left on DAY 4', 'args': [1, 4], 'kind': 'action', 'want': 14, 'tol': 0}, {'about': '5 seats left on DAY 3', 'args': [5, 3], 'kind': 'action', 'want': 5, 'tol': 0}], 'pick_action': [{'about': 'my_table in row 0, with the ball at 0.2882', 'args': [[[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]], 0], 'kind': 'action', 'want': 0, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': 'my_table in row 0, with the ball at 0.7343', 'args': [[[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]], 0], 'kind': 'action', 'want': 0, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [2]]]}, {'about': 'my_table in row 8, with the ball at 0.9236', 'args': [[[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]], 8], 'kind': 'action', 'want': 1, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [4]]]}, {'about': 'my_table in row 8, with the ball at 0.6271', 'args': [[[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]], 8], 'kind': 'action', 'want': 1, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [1]]]}, {'about': 'my_table in row 12, with the ball at 0.1563', 'args': [[[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]], 12], 'kind': 'action', 'want': 0, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [8]]]}, {'about': "the start's probabilities in row 11, with the ball at 0.4480", 'args': [[[0.9998989708060922, 0.00010102919390777272], [0.6899744811276125, 0.3100255188723876], [0.2141650169574414, 0.7858349830425586], [0.37754066879814546, 0.6224593312018546], [0.289050497374996, 0.7109495026250039], [0.9999924954984029, 7.504501597109999e-06], [0.5249791874789399, 0.47502081252106], [0.9983411989198255, 0.0016588010801744213], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106], [0.005486298899450404, 0.9945137011005495], [0.574442516811659, 0.425557483188341], [2.2544429650445413e-05, 0.9999774555703496], [0.7685247834990178, 0.23147521650098232], [0.37754066879814546, 0.6224593312018546]], 11], 'kind': 'action', 'want': 0, 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [14]]]}], 'two_days': [{'about': 'my_table, on luck 1 of the check', 'args': [[[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]]], 'kind': 'numbers', 'want': [0, 5], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': 'my_table, on luck 2 of the check', 'args': [[[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]]], 'kind': 'numbers', 'want': [0, 0], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [3]]]}, {'about': 'my_table, on luck 3 of the check', 'args': [[[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]]], 'kind': 'numbers', 'want': [0, 6], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [7]]]}, {'about': 'a policy that always plays STANDARD, [0, 1] in every row, on luck 4 of the check', 'args': [[[0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1]]], 'kind': 'numbers', 'want': [6, 4], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [9]]]}, {'about': "the start's probabilities, on luck 5 of the check", 'args': [[[0.9998989708060922, 0.00010102919390777272], [0.6899744811276125, 0.3100255188723876], [0.2141650169574414, 0.7858349830425586], [0.37754066879814546, 0.6224593312018546], [0.289050497374996, 0.7109495026250039], [0.9999924954984029, 7.504501597109999e-06], [0.5249791874789399, 0.47502081252106], [0.9983411989198255, 0.0016588010801744213], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106], [0.005486298899450404, 0.9945137011005495], [0.574442516811659, 0.425557483188341], [2.2544429650445413e-05, 0.9999774555703496], [0.7685247834990178, 0.23147521650098232], [0.37754066879814546, 0.6224593312018546]]], 'kind': 'numbers', 'want': [5, 6], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [5]]]}], 'play_flight': [{'about': 'my_table, on luck 1 of the check', 'args': [[[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]]], 'kind': 'numbers', 'want': [0, 5, 0, 6], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': 'my_table, on luck 2 of the check', 'args': [[[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]]], 'kind': 'numbers', 'want': [4, 0, 0, 5], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [4]]]}, {'about': 'my_table, on luck 3 of the check', 'args': [[[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]]], 'kind': 'numbers', 'want': [0, 4, 2, 5], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [6]]]}, {'about': 'a policy that always plays PREMIUM, [1, 0] in every row, on luck 4 of the check', 'args': [[[1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0], [1, 0]]], 'kind': 'numbers', 'want': [5, 5, 0, 0], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [8]]]}, {'about': 'a policy that always plays STANDARD, [0, 1] in every row, on luck 5 of the check', 'args': [[[0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1], [0, 1]]], 'kind': 'numbers', 'want': [6, 4], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [10]]]}, {'about': "the start's probabilities, on luck 6 of the check", 'args': [[[0.9998989708060922, 0.00010102919390777272], [0.6899744811276125, 0.3100255188723876], [0.2141650169574414, 0.7858349830425586], [0.37754066879814546, 0.6224593312018546], [0.289050497374996, 0.7109495026250039], [0.9999924954984029, 7.504501597109999e-06], [0.5249791874789399, 0.47502081252106], [0.9983411989198255, 0.0016588010801744213], [0.401312339887548, 0.598687660112452], [0.52497918747894, 0.47502081252106], [0.005486298899450404, 0.9945137011005495], [0.574442516811659, 0.425557483188341], [2.2544429650445413e-05, 0.9999774555703496], [0.7685247834990178, 0.23147521650098232], [0.37754066879814546, 0.6224593312018546]]], 'kind': 'numbers', 'want': [0, 5, 0, 6], 'tol': 0, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}], 'episode_return': [{'about': 'the rewards [5, 0, 4, 6]', 'args': [[5, 0, 4, 6]], 'kind': 'number', 'want': 15, 'tol': 1e-09, 'keep': [0]}, {'about': 'the rewards [2, 4]', 'args': [[2, 4]], 'kind': 'number', 'want': 6, 'tol': 1e-09, 'keep': [0]}, {'about': 'the rewards [6]', 'args': [[6]], 'kind': 'number', 'want': 6, 'tol': 1e-09, 'keep': [0]}, {'about': 'the rewards []', 'args': [[]], 'kind': 'number', 'want': 0, 'tol': 1e-09, 'keep': [0]}, {'about': 'the rewards [5, 5, 5, 5]', 'args': [[5, 5, 5, 5]], 'kind': 'number', 'want': 20, 'tol': 1e-09, 'keep': [0]}]}
_N2_SLIPS = {'row_of': [{'answers': [0, 4, 2, 3, 9, 0], 'hint': 'a situation matches when both numbers match: situation[0] == seats_left and situation[1] == day'}, {'answers': [1, 5, 7, 13, 15, 6], 'hint': 'count from 0: the first row, DAY 1 with 5 seats left, is row 0'}], 'pick_action': [{'answers': [0, 0, 1, 0, 0, 0], 'hint': "take the row of today's situation, policy[row]"}, {'answers': [0, 0, 0, 0, 1, 0], 'hint': 'throw the ball once: the given line ball = spin() does it, so land the ball you have, ball'}, {'answers': [[0.8, 1.0], [0.8, 1.0], [0.4, 1.0], [0.4, 1.0], [0.2, 1.0], [0.574442516811659, 1.0]], 'hint': 'give back where the ball lands, where_it_lands(ends, ball)'}], 'two_days': [{'answers': [[0, 5], [0, 0], [0, 6], [6, 4], [5, 5]], 'hint': 'DAY 2 starts from the seats DAY 1 left: row_of(seats_left, 2)'}, {'answers': [[0, 2], [0, 5], [0, 4], [6, 4], [5, 2]], 'hint': "the line above already picked the day's price: call sell(seats_left, price), so that each day throws one ball"}], 'play_flight': [{'answers': [[0], [4, 0], [0], None, [6, 6, 6, 6], [0]], 'hint': 'seats_left goes down by the seats sold: seats_left - seats_sold'}, {'answers': [None, None, [0, 0, 0, 5], None, None, None], 'hint': "the situation of today has today's day: row = row_of(seats_left, day)"}, {'answers': [[0, 1, 0, 3], [2, 0, 0, 1], [0, 2, 1, 1], [1, 1, 0, 0], [3, 2], [0, 1, 0, 3]], 'hint': "a reward is the day's money: rewards.append(money)"}], 'episode_return': [{'answers': [6, 4, 6, 0, 5], 'hint': 'add to what the total already holds: total = total + money'}, {'answers': [5, 2, 6, 0, 5], 'hint': 'give the total back after the loop: return total, as far left as for'}]}
_N2_SUCCESS = {'row_of': 'row_of works: DAY 1 with 5 seats left is row 0, DAY 4 with 1 seat left row 14.', 'pick_action': 'pick_action works: every ball the check threw gave the price of its slot.', 'two_days': 'two_days works: DAY 2 starts from the seats that DAY 1 left.', 'play_flight': 'play_flight works: on every luck of the check, your flights follow the plane day by day.', 'episode_return': 'episode_return works: the rewards [5, 0, 4, 6] give the return $15.'}
_N2_DEFAULT = {'row_of': 'compare both numbers of the situation: situation[0] == seats_left and situation[1] == day; give back row, the counter, and count from 0', 'pick_action': 'probabilities is policy[row], ends is make_roulette(probabilities), and the price is where_it_lands(ends, ball)', 'two_days': 'DAY 2 is three lines: row = row_of(seats_left, 2), price = pick_action(policy, row), then seats_sold, money_2 = sell(seats_left, price)', 'play_flight': 'in the loop: row = row_of(seats_left, day), price = pick_action(policy, row), seats_sold, money = sell(seats_left, price), rewards.append(money), then seats_left = seats_left - seats_sold', 'episode_return': 'total = total + money in the loop, then return total'}
_N2_KEEP_HINT = {'row_of': 'row_of reads SITUATIONS and leaves it as it is', 'pick_action': 'pick_action reads the probability table and leaves it as it is', 'two_days': 'two_days reads the probability table and leaves it as it is', 'play_flight': 'play_flight reads the probability table and leaves it as it is', 'episode_return': 'episode_return reads the rewards and leaves them as they are'}
_N2_TASKS = {'row_of': 'task A', 'pick_action': 'task B', 'two_days': 'task C', 'play_flight': 'task D', 'episode_return': 'task E'}
_N2_NEEDS = {'row_of': [], 'pick_action': ['make_roulette', 'where_it_lands'], 'two_days': ['row_of', 'pick_action'], 'play_flight': ['row_of', 'pick_action'], 'episode_return': []}
_N2_LONG = []
_N2_NONE_HINT = {'row_of': 'row_of gives None when no situation matches: situation[0] is the seats left and situation[1] the day, as in [5, 1]'}
_N2_DICE_OF = 'spin'
_N2_GAPS = {}
_N2_WORKING = {}
_N2_DICE = {}


def _n2_num(x):
    """A number with at most 4 decimals: 154.3961, 2.585, 218.5."""
    if isinstance(x, bool) or not isinstance(x, (int, float)):
        return repr(x)
    if x != x or x in (float("inf"), float("-inf")):
        return repr(x)
    if isinstance(x, int):
        return str(x)
    text = ("%.4f" % x).rstrip("0").rstrip(".")
    if text == "-0":
        text = "0"
    if text == "0" and x != 0:
        text = "%.2g" % x
    return text


def _n2_text(value):
    """Numbers with at most 4 decimals, lists of them in brackets."""
    if isinstance(value, (list, tuple)):
        parts = []
        for x in value:
            parts.append(_n2_text(x))
        return "[" + ", ".join(parts) + "]"
    return _n2_num(value)


def _n2_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = _n2_text(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 72:
        text = text[:69] + "..."
    return text


def _n2_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _n2_same(kind, got, want, tol):
    """True when got is the right answer want, within tol."""
    try:
        if kind == "number":
            return _n2_is_number(got) and abs(got - want) <= tol
        if kind == "action":
            return _n2_is_number(got) and got == want
        if kind in ("numbers", "table"):
            if not isinstance(got, (list, tuple)) or len(got) != len(want):
                return False
            for k in range(len(want)):
                if not _n2_same("numbers" if kind == "table" else "number", got[k], want[k], tol):
                    return False
            return True
    except Exception:
        return False
    return False


def _n2_like(got, answer, tol):
    """True when got has the shape and the numbers of a slip's answer, within tol: a slip may give a number where a
    list is due (return total), or a list of numbers where a list of lists is due."""
    if isinstance(answer, (list, tuple)):
        if not isinstance(got, (list, tuple)) or len(got) != len(answer):
            return False
        for k in range(len(answer)):
            if not _n2_like(got[k], answer[k], tol):
                return False
        return True
    return _n2_is_number(got) and _n2_is_number(answer) and abs(got - answer) <= tol


def _n2_copy(value):
    """A deep copy of lists of lists of numbers."""
    if isinstance(value, list):
        out = []
        for x in value:
            out.append(_n2_copy(x))
        return out
    return value


def _n2_dice(seed, first, count):
    """The dice of count episodes of a seed from episode first, rolled by the kit once per seed and kept."""
    rolled = _N2_DICE.get(seed)
    if rolled is None or len(rolled) < first + count:
        if len(_N2_DICE) > 3:
            _N2_DICE.clear()
        rolled = globals()[_N2_DICE_OF](seed, max(first + count, 10))
        _N2_DICE[seed] = rolled
    return rolled[first:first + count]


def _n2_args(test):
    """Fresh inputs for a test: copies of the lists, and the dice it names."""
    args = []
    for a in test["args"]:
        if isinstance(a, dict) and "dice" in a:
            args.append(_n2_dice(a["dice"][0], a["dice"][1], a["dice"][2]))
        elif isinstance(a, dict) and "episode" in a:
            args.append(_n2_dice(a["episode"][0], a["episode"][1], 1)[0])
        else:
            args.append(_n2_copy(a))
    return args


def _n2_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N2_GAPS:
        return _N2_GAPS[code]
    # A gap such as "total = ..." stays in the compiled function; a gap such as "if ...:" only in the source, which
    # Jupyter and Colab keep for every cell
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N2_GAPS[code] = gap
    return gap


def _n2_holds_gap(value):
    """True when a value, or a list inside it, still holds a ... (a gap of a cell above)."""
    if value is Ellipsis:
        return True
    if isinstance(value, (list, tuple)):
        for x in value:
            if _n2_holds_gap(x):
                return True
    return False


def _n2_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _n2_key(name, seen=()):
    """The code of the function called name and of the functions it needs: a new version of any of them is checked
    again."""
    if name in seen:
        return ()
    fn = globals().get(name)
    key = [getattr(fn, "__code__", None)]
    for other in _N2_NEEDS.get(name, []):
        key.extend(_n2_key(other, seen + (name,)))
    return tuple(key)


def _n2_call(fn, args):
    """Run fn with its prints silenced: {"answer", "error" (None when the call went through), "line" (the line of
    your code where it happened)}. Never raises."""
    import contextlib, io, traceback
    run = {"answer": None, "error": None, "line": None}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            run["answer"] = fn(*args)
    except Exception as error:
        run["error"] = "%s: %s" % (type(error).__name__, error)
        try:
            skip = (_n2_call.__code__.co_filename, globals()[_N2_DICE_OF].__code__.co_filename)
            for frame in traceback.extract_tb(error.__traceback__):
                if frame.filename not in skip and frame.line:
                    run["line"] = frame.line.strip()
        except Exception:
            run["line"] = None
    return run


def _n2_explain(run):
    """A hint for an error."""
    text = run["error"]
    if "NoneType" in text:
        return 'a value in that line is None: a function it calls gave nothing back. row_of gives None for a situation that is not in SITUATIONS: call it as row_of(seats_left, day)'
    if "ellipsis" in text:
        return "a ... is still in a function that this one calls: finish that function first"
    if "list indices must be integers" in text:
        return 'a place in [ ] must be a whole number: a row or a price, such as policy[row] for the row of a situation and probabilities[price] for one of its numbers'
    if "is not subscriptable" in text:
        return "that line uses [ ] on a single number: only a list has places to look up"
    if "is not iterable" in text:
        return ("a single number stands where a list was expected: check the names on the left of each = "
                "(update_preference changes the table itself and gives back only the new preference)")
    if 'sell needs' in text:
        return 'the plane got a number of seats or a price it does not know: check what pick_action gives back, and that seats_left goes down by the seats sold'
    if "ZeroDivisionError" in text:
        return "a division by zero: divide by a number that cannot be 0, such as the total or len(...)"
    if "NameError" in text:
        return "a name in that line is not defined: check its spelling, and run the cells above it first"
    if "can only concatenate" in text or "unsupported operand" in text:
        return "that line mixes a list and a number: take one number of the list with [ ] first"
    if "positional argument" in text:
        return "a function got more or fewer inputs than it takes: compare the call with the function's def line"
    if "out of range" in text:
        return "a place in [ ] is past the end of the list: count from 0, the last place is len(...) - 1"
    return "read that line again, compare it with the hints, run the function's cell, then this check again"


def _n2_failure(name, fn):
    """None when fn passes every test of its task, else (head, what, hint). The kit's state (its luck) is put back
    afterwards when the kit can save it."""
    save = globals().get("_kit_save")
    saved = save() if callable(save) else None
    try:
        return _n2_failure_of(name, fn)
    finally:
        if saved is not None:
            globals()["_kit_restore"](saved)


def _n2_failure_of(name, fn):
    tests = _N2_TESTS[name]
    for k in range(len(tests)):
        test = tests[k]
        args = _n2_args(test)
        kept = _n2_copy(args)
        for setup in test.get("setup", []):
            globals()[setup[0]](*setup[1])
        run = _n2_call(fn, args)
        head = "On " + test["about"]
        if run["error"] is not None:
            where = ""
            if run["line"]:
                where = ", in the line  %s " % run["line"]
            return (head, "%s stopped with an error, %s%s" % (name, run["error"], where), _n2_explain(run))
        got = run["answer"]
        if not _n2_same(test["kind"], got, test["want"], test["tol"]):
            if got is None:
                return (head, "%s gave None" % name, _N2_NONE_HINT.get(
                    name, "a function gives its answer with return: keep the return line at its end"))
            hint = _N2_DEFAULT[name]
            for slip in _N2_SLIPS[name]:
                answer = slip["answers"][k]
                if answer is not None and _n2_like(got, answer, test["tol"]):
                    hint = slip["hint"]
                    break
            return (head, "%s should give %s, and yours gives %s" % (name, _n2_short(test["want"]),
                                                                    _n2_short(got)), hint)
        for i in test.get("keep", []):
            if args[i] != kept[i]:
                return (head, "%s changed its input %s" % (name, _n2_short(kept[i])), _N2_KEEP_HINT[name])
        if test.get("table"):
            t = test["table"]
            if not _n2_same("table", args[t["arg"]], t["want"], test["tol"]):
                return (head, "after %s the table of preferences should be %s, and it is %s" % (
                    name, _n2_short(t["want"]), _n2_short(args[t["arg"]])), _N2_KEEP_HINT[name])
    return None


def _n2_works(name):
    """True when the function called name exists, is finished, the functions it uses work, and it passes its tests.
    Remembered for each version of it."""
    fn = globals().get(name)
    if not callable(fn) or getattr(fn, "__code__", None) is None or _n2_has_gap(fn):
        return False
    for other in _N2_NEEDS.get(name, []):
        if _n2_uses(fn, other) and not _n2_works(other):
            return False
    if name not in _N2_TESTS:
        return True
    key = _n2_key(name)
    if key not in _N2_WORKING:
        _N2_WORKING[key] = _n2_failure(name, fn) is None
    return _N2_WORKING[key]


def _n2_waits(name, fn, cell):
    """One calm line and True when fn cannot run yet: it is not finished, or a function it uses does not work yet."""
    task = ""
    if name in _N2_TASKS:
        task = " (%s)" % _N2_TASKS[name]
    if _n2_has_gap(fn):
        print("⏳ %s is not finished yet%s: replace each ... with your code, run its cell, then %s again."
              % (name, task, cell))
        return True
    for other in _N2_NEEDS.get(name, []):
        if not _n2_uses(fn, other) or _n2_works(other):
            continue
        if other in _N2_TASKS:
            needed = globals().get(other)
            what = "does not work yet"
            if callable(needed) and _n2_has_gap(needed):
                what = "is not finished yet"
            print("⏳ %s uses your %s (%s), which %s. Get check_%s(%s) to ✅ first, then run %s again."
                  % (name, other, _N2_TASKS[other], what, other, other, cell))
            return True
        # a given function: it waits for the tasks it uses
        for deeper in _N2_NEEDS.get(other, []):
            if deeper in _N2_TASKS and not _n2_works(deeper):
                print("⏳ %s uses %s, which uses your %s (%s). Get check_%s(%s) to ✅ first, then run %s again."
                      % (name, other, deeper, _N2_TASKS[deeper], deeper, deeper, cell))
                return True
        print("⏳ %s uses %s, which does not work yet: run its cell again, then %s." % (name, other, cell))
        return True
    return False


def _n2_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    if _n2_waits(name, fn, "this check"):
        return
    failure = _n2_failure(name, fn)
    if failure is None:
        print("✅ " + _N2_SUCCESS[name])
        return
    head, what, hint = failure
    print("❌ %s: %s." % (head, what))
    print("   💡 Hint: " + hint + ".")


def _n2_safe_check(name, fn):
    try:
        _n2_check(name, fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_row_of(fn):
    """Task a: the rows of several situations, the first and the last among them"""
    _n2_safe_check("row_of", fn)

def check_pick_action(fn):
    """Task b: the price of several rows and tables, each on a ball the check throws"""
    _n2_safe_check("pick_action", fn)

def check_two_days(fn):
    """Task c: two days of several policies, each on luck the check deals"""
    _n2_safe_check("two_days", fn)

def check_play_flight(fn):
    """Task d: flights of several policies, each on luck the check deals"""
    _n2_safe_check("play_flight", fn)

def check_episode_return(fn):
    """Task e: the return of several flights"""
    _n2_safe_check("episode_return", fn)


def play(fn, *inputs):
    """Runs one of your functions and gives its answer: play(average_return, preferences) is
    average_return(preferences). When the function is not finished yet, uses a function that does not work yet, gets
    None from a cell above or stops with an error, play prints one calm line and gives None, so the cells below
    still run."""
    import time
    try:
        if not callable(fn):
            print("⏳ play needs your function itself, as in play(average_return, preferences).")
            return None
        name = getattr(fn, "__name__", "your function")
        if _n2_waits(name, fn, "this cell"):
            return None
        for x in inputs:
            if x is None:
                print("⏳ %s waits for a value that a cell above did not give yet (its ⏳ line says why)." % name)
                return None
            if _n2_holds_gap(x):
                print("⏳ %s waits for a value that still holds a ...: fill it in above, run its cell, then this cell."
                      % name)
                return None
        if name in _N2_LONG and not _n2_works(name):
            print("⏳ %s waits until it passes its check: get check_%s(%s) to ✅ first, then run this cell again."
                  % (name, name, name))
            return None
        long = name in _N2_LONG
        if long and callable(globals().get("_kit_before_long")):
            globals()["_kit_before_long"]()
        start = time.time()
        try:
            answer = fn(*inputs)
        except Exception as error:
            print("❌ %s stopped with an error, %s: %s. Its check says more." % (name, type(error).__name__, error))
            return None
        finally:
            if long and callable(globals().get("_kit_after_long")):
                globals()["_kit_after_long"]()
        seconds = time.time() - start
        if seconds > 5:
            print("⏱️ %s took %d seconds." % (name, round(seconds)))
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the folded cells at the top again, then this cell." % error)
        return None

_N2_TABLE = [[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]]
_N2_START_PREFERENCES = [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]
_N2_READ_ROW = 8
_N2_MY = [0, 1, 5]


def _n2_ready(name, cell):
    """True when your function called name exists and passes its check; else one calm line."""
    fn = globals().get(name)
    if not callable(fn):
        print("⏳ %s is not there yet: run the cell of %s first, then %s." % (name, _N2_TASKS.get(name, name), cell))
        return False
    if _n2_waits(name, fn, cell):
        return False
    if not _n2_works(name):
        print("⏳ This cell waits until %s passes its check: get check_%s(%s) to ✅ first, then run %s again."
              % (name, name, name, cell))
        return False
    return True


def _n2_given_ready(names, cell):
    """True when the given functions called names exist; else one calm line."""
    for name in names:
        if not callable(globals().get(name)):
            print("⏳ %s is not there yet: run the folded cells at the top and the given cells above, then %s."
                  % (name, cell))
            return False
    return True


def _n2_policy_ready(policy, cell):
    """True when policy is a probability table without gaps; else one calm line."""
    if policy is None or _n2_holds_gap(policy):
        print("⏳ %s waits for a probability table: finish my_table first (warm-up 1), then run it again." % cell)
        return False
    return True


def check_table(table):
    """Warm-up 1: my_table, the made-up probability table."""
    try:
        if _n2_holds_gap(table):
            print("⏳ my_table is not finished yet (warm-up 1): replace each ... with a row of two probabilities, run "
                  "its cell, then this check again.")
            return
        if not isinstance(table, list) or len(table) != len(SITUATIONS):
            print("❌ my_table should be a list of %d rows, one per situation, and yours has %s." % (
                len(SITUATIONS), len(table) if isinstance(table, list) else _n2_short(table)))
            print("   💡 Hint: write one row per line, each in [ ] and followed by a comma, all inside the outer [ ].")
            return
        for r in range(len(SITUATIONS)):
            row = table[r]
            if not _n2_same("numbers", row, _N2_TABLE[r], 1e-9):
                print("❌ Row %d, %s, should be %s, and yours is %s." % (r, _kit_situation(r), _n2_text(_N2_TABLE[r]),
                                                                       _n2_short(row)))
                hint = "copy the row from the table above, two numbers in the order PREMIUM, STANDARD"
                if isinstance(row, list) and len(row) == 2 and _n2_same("numbers", [row[1], row[0]], _N2_TABLE[r],
                                                                        1e-9):
                    hint = "the numbers of a row go in the order of the columns: PREMIUM, STANDARD"
                print("   💡 Hint: " + hint + ".")
                return
        print("✅ my_table is the made-up probability table: %d rows, one per situation, and each row adds up to 1."
              % len(SITUATIONS))
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_reading(probability, row):
    """Warm-up 2: a number and a row read from my_table."""
    try:
        if probability is Ellipsis or _n2_holds_gap(row):
            print("⏳ Warm-up 2 is not finished yet: replace each ... with your code, run its cell, then this check "
                  "again.")
            return
        want_probability = _N2_TABLE[_N2_READ_ROW][STANDARD]
        want_row = _N2_TABLE[0]
        if not _n2_same("number", probability, want_probability, 1e-9):
            print("❌ probability should be %s, the probability of STANDARD in row %d, and yours is %s."
                  % (_n2_text(want_probability), _N2_READ_ROW, _n2_short(probability)))
            hint = "the row comes first, then the column: my_table[%d][STANDARD]" % _N2_READ_ROW
            if _n2_same("number", probability, _N2_TABLE[_N2_READ_ROW][PREMIUM], 1e-9):
                hint = "that is the other column of row %d: STANDARD is the column 1" % _N2_READ_ROW
            print("   💡 Hint: " + hint + ".")
            return
        if not _n2_same("numbers", row, want_row, 1e-9):
            print("❌ row should be row 0, %s, and yours is %s." % (_n2_text(want_row), _n2_short(row)))
            print("   💡 Hint: a row is the table with one [ ]: my_table[0].")
            return
        print("✅ Both right: my_table[%d][STANDARD] is %s, and my_table[0] is %s." % (
            _N2_READ_ROW, _n2_text(want_probability), _n2_text(want_row)))
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_day(seats_sold, money):
    """Warm-up 3: one day at STANDARD with 5 seats left, sell(5, STANDARD)."""
    try:
        if seats_sold is None and money is None:
            print("⏳ Warm-up 3 is not finished yet: replace each ... with your code, run its cell, then this check "
                  "again.")
            return
        for count, probability in _KIT_BUYERS[STANDARD]:
            if seats_sold == min(count, SEATS) and money == DOLLARS[STANDARD] * min(count, SEATS):
                print("✅ A day at STANDARD with 5 seats left: %d %s sold, %s." % (
                    seats_sold, "seat" if seats_sold == 1 else "seats", _kit_dollars(money)))
                return
        print("❌ A day at STANDARD sells 1, 2 or 3 seats at %s each; your day gave %s seats sold and the money %s."
              % (_kit_dollars(DOLLARS[STANDARD]), _n2_short(seats_sold), _n2_short(money)))
        hint = "call sell with the seats left first, then the price: sell(5, STANDARD)"
        if [seats_sold, money] in ([0, 0], [1, DOLLARS[PREMIUM]]):
            hint = "that is a day at PREMIUM: the price of this warm-up is STANDARD"
        print("   💡 Hint: " + hint + ".")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_my_preferences(preferences):
    """Warm-up 1 of part 2: the preferences at random with the preference for STANDARD in row 0 set to 5."""
    try:
        r0, p0, v0 = _N2_MY
        if preferences is None or _n2_holds_gap(preferences):
            print("⏳ Warm-up 1 is not finished yet: replace the ... with your code, run its cell, then this check "
                  "again.")
            return
        if not _kit_is_table(preferences):
            print("❌ my_preferences should be a table of preferences, %d rows of two numbers, and it is %s." % (
                len(SITUATIONS), _n2_short(preferences)))
            print("   💡 Hint: start from my_preferences = start_preferences(), then change one preference.")
            return
        changed = []
        for r in range(len(SITUATIONS)):
            for p in range(2):
                if preferences[r][p] != _N2_START_PREFERENCES[r][p]:
                    changed.append([r, p])
        if len(changed) == 0:
            print("⏳ my_preferences is still the table at random: set the preference for STANDARD in row 0 to 5 in the "
                  "cell of warm-up 1, run it, then this check again.")
            return
        if changed == [[r0, p0]] and preferences[r0][p0] == v0:
            print("✅ The preference for STANDARD in row 0 is 5: the probability of STANDARD on DAY 1 went from %.2f to "
                  "%.2f." % (
                _n2_probabilities(_N2_START_PREFERENCES[r0])[p0], _n2_probabilities(preferences[r0])[p0]))
            return
        if len(changed) > 1:
            print("❌ %d preferences of my_preferences differ from the table at random; warm-up 1 changes one."
                  % len(changed))
            print("   💡 Hint: start again from my_preferences = start_preferences(), then change only "
                  "my_preferences[0][STANDARD].")
            return
        r, p = changed[0]
        if [r, p] != [r0, p0]:
            print("❌ You changed the preference for %s in row %d; warm-up 1 changes the preference for STANDARD in row "
                  "0." % (
                PRICES[p], r))
            print("   💡 Hint: the row comes first, then the column: my_preferences[0][STANDARD] = 5.")
            return
        print("❌ The preference for STANDARD in row 0 is %s; warm-up 1 sets it to 5." % _n2_short(preferences[r][p]))
        print("   💡 Hint: my_preferences[0][STANDARD] = 5.")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def _n2_probabilities(row):
    largest = max(row)
    powers = []
    for x in row:
        powers.append(exp(x - largest))
    total = 0
    for x in powers:
        total = total + x
    out = []
    for x in powers:
        out.append(x / total)
    return out


def throw_balls(probabilities, throws):
    """Throws the ball throws times on the roulette of a row of probabilities, with make_roulette and where_it_lands,
    and prints how often each price came."""
    try:
        if probabilities is None or _n2_holds_gap(probabilities):
            print("⏳ throw_balls waits for a row of probabilities: finish my_table first (warm-up 1), then run it "
                  "again.")
            return
        if not _n2_given_ready(["make_roulette", "where_it_lands"], "this cell"):
            return
        ends = make_roulette(probabilities)
        counts = [0, 0]
        for k in range(throws):
            slot = where_it_lands(ends, spin())
            counts[slot] = counts[slot] + 1
        print("In %s throws:   PREMIUM %d times, STANDARD %d times" % (format(throws, ","), counts[0], counts[1]))
        print("Their shares:      PREMIUM %.2f, STANDARD %.2f" % (counts[0] / throws, counts[1] / throws))
        print("The probabilities: PREMIUM %.2f, STANDARD %.2f" % (probabilities[0], probabilities[1]))
    except Exception as error:
        print("❌ throw_balls could not run (%s). Run the folded cells at the top again, then this cell." % error)


def show_picks(policy, picks):
    """Your pick_action, picks times in each row of a probability table: how often each price came, and the
    shares."""
    try:
        if not _n2_policy_ready(policy, "show_picks"):
            return
        if not _n2_ready("pick_action", "this cell"):
            return
        counts = []
        shares = []
        for r in range(len(SITUATIONS)):
            counts.append([0, 0])
            for k in range(picks):
                price = pick_action(policy, r)
                counts[r][price] = counts[r][price] + 1
            shares.append([counts[r][0] / picks, counts[r][1] / picks])
        print("Your pick_action, %s times in each row: the shares of each price, to set beside the probability table"
              % format(picks, ","))
        show_probabilities(shares)
    except Exception as error:
        print("❌ show_picks could not run (%s). Run the folded cells at the top again, then this cell." % error)


def count_picks(how_many):
    """Your pick_preference, how_many times: how often each preference came."""
    try:
        if not _n2_ready("pick_preference", "this cell"):
            return
        counts = []
        for r in range(len(SITUATIONS)):
            counts.append([0, 0])
        for k in range(how_many):
            row, price = pick_preference()
            counts[row][price] = counts[row][price] + 1
        print("Your pick_preference, %s times: how often each preference came" % format(how_many, ","))
        show_counts(counts)
    except Exception as error:
        print("❌ count_picks could not run (%s). Run the folded cells at the top again, then this cell." % error)


def show_flight(policy):
    """Plays one flight with your row_of and pick_action and the kit's sell, and prints it day by day."""
    try:
        if not _n2_policy_ready(policy, "show_flight"):
            return
        if not _n2_ready("row_of", "this cell") or not _n2_ready("pick_action", "this cell"):
            return
        seats_left = SEATS
        total = 0
        print("day     situation             price     sold   money")
        for day in range(1, DAYS + 1):
            if seats_left > 0:
                row = row_of(seats_left, day)
                price = pick_action(policy, row)
                seats_sold, money = sell(seats_left, price)
                print("DAY %d   %-20s  %-8s  %4d   %5s" % (day, _kit_situation(row), PRICES[price], seats_sold,
                                                          _kit_dollars(money)))
                total = total + money
                seats_left = seats_left - seats_sold
        if seats_left == 0:
            print("FULL FLIGHT!   MONEY %s" % _kit_dollars(total))
        else:
            print("TAKEOFF! %d EMPTY %s.   MONEY %s" % (seats_left, "SEAT" if seats_left == 1 else "SEATS",
                                                       _kit_dollars(total)))
    except Exception as error:
        print("❌ show_flight could not run (%s). Run the folded cells at the top again, then this cell." % error)

print("✅ The checks of part 1 are ready.")

## 1. The situations

Every day you set one price for the seats that are left:

- **PREMIUM**, \$5 a seat: no buyer on 60% of the days, one buyer on 40%.
- **STANDARD**, \$2 a seat: one buyer on 20% of the days, two on 40%, three on 40%.

The buyers take the seats that are left. A flight ends with a FULL FLIGHT when the last seat is sold, or at
takeoff after DAY 4, with its EMPTY SEATS. The game never shows these probabilities; the kit knows them.

A policy looks at the situation of the day: the seats left and the day, a pair of numbers `[seats left, day]`.
Every flight starts in `[5, 1]`: 5 seats left on DAY 1. The kit keeps every situation a flight can be in, in
`SITUATIONS`, one per row of a table.

In [ ]:
# The situations a flight can be in, one per row
show_situations()

**Reading the result.** STANDARD sells at most 3 seats in a day, PREMIUM at most 1. So on DAY 2 the plane has 5,
4, 3 or 2 seats left, and on DAY 3 and DAY 4 anything from 1 to 5: 15 situations. The arcade's board has 20
squares; the other 5 are squares no flight ever reaches, and they get no row.

## 2. The probability table

A stochastic policy is a probability table, as on page 3 of Old Bessie: a row for each situation, a column
for each price, and each row adds up to 1. Here is a made-up one, where every situation of a day has
the same row:

| | PREMIUM | STANDARD |
|---|---|---|
| **DAY 1** (rows 0) | 0.8 | 0.2 |
| **DAY 2** (rows 1 to 4) | 0.6 | 0.4 |
| **DAY 3** (rows 5 to 9) | 0.4 | 0.6 |
| **DAY 4** (rows 10 to 14) | 0.2 | 0.8 |

On DAY 2, this policy plays PREMIUM on 6 days out of 10 on average, and STANDARD on 4.

In Python, a table is a list of rows, and each row is a list of two numbers in the order of the columns:
PREMIUM, STANDARD. The rows come in the order of `SITUATIONS`. Rows 0 to 11 are written for you.

### 🎯 Warm-up 1. Write the table in Python

In [ ]:
# ✅ SOLUTION, warm-up 1: the made-up probability table, one row per situation
my_table = [
    [0.8, 0.2],        # row 0: DAY 1, 5 SEATS LEFT. Columns: PREMIUM, STANDARD
    [0.6, 0.4],        # row 1: DAY 2, 5 SEATS LEFT
    [0.6, 0.4],        # row 2: DAY 2, 4 SEATS LEFT
    [0.6, 0.4],        # row 3: DAY 2, 3 SEATS LEFT
    [0.6, 0.4],        # row 4: DAY 2, 2 SEATS LEFT
    [0.4, 0.6],        # row 5: DAY 3, 5 SEATS LEFT
    [0.4, 0.6],        # row 6: DAY 3, 4 SEATS LEFT
    [0.4, 0.6],        # row 7: DAY 3, 3 SEATS LEFT
    [0.4, 0.6],        # row 8: DAY 3, 2 SEATS LEFT
    [0.4, 0.6],        # row 9: DAY 3, 1 SEAT LEFT
    [0.2, 0.8],        # row 10: DAY 4, 5 SEATS LEFT
    [0.2, 0.8],        # row 11: DAY 4, 4 SEATS LEFT
    [0.2, 0.8],        # row 12: DAY 4, 3 SEATS LEFT
    [0.2, 0.8],        # row 13: DAY 4, 2 SEATS LEFT
    [0.2, 0.8],        # row 14: DAY 4, 1 SEAT LEFT
]
show_probabilities(my_table)

In [ ]:
#@title 💡 Hint 1 for warm-up 1 (double-click to open) { display-mode: "form" }
# Each ... becomes one row: two numbers in [ ], separated by a comma, copied from the table above.
# Rows 12, 13 and 14 are situations of DAY 4. Keep the comma after each row: the rows are the items of one
# list.

In [ ]:
#@title 💡 Hint 2 for warm-up 1 (double-click to open) { display-mode: "form" }
# DAY 4's row reads from the table: PREMIUM 0.2, STANDARD 0.8. So each of the three lines is
#     [0.2, 0.8],

In [ ]:
# Check warm-up 1
check_table(my_table)

### Reading a table

The kit gives each price its number, counted from 0: `PREMIUM` is 0 and `STANDARD` 1. So `my_table[8]` is row
8, DAY 3 with 2 seats left, and `my_table[8][PREMIUM]` is one number of that row: the probability of PREMIUM in
that situation. The row comes first, then the column.

In [ ]:
# A row, then one number of it
print("Row 8:", my_table[8])
print("The probability of PREMIUM in row 8:", my_table[8][PREMIUM])

### 🎯 Warm-up 2. Read your table

Put the probability of STANDARD in row 8 into `probability`, and row 0 into `row`.

In [ ]:
# ✅ SOLUTION, warm-up 2: read your table
probability = my_table[8][STANDARD]     # row 8, then its number in the column of STANDARD
row = my_table[0]                       # row 0

In [ ]:
#@title 💡 Hint 1 for warm-up 2 (double-click to open) { display-mode: "form" }
# A number of the table takes two [ ]: first the row, then the column, the price.
# A row takes one [ ].

In [ ]:
#@title 💡 Hint 2 for warm-up 2 (double-click to open) { display-mode: "form" }
#     probability = my_table[8][STANDARD]
#     row = my_table[0]

In [ ]:
# Check warm-up 2
check_reading(probability, row)

## 3. The row of a situation

During a flight you know the seats left and the day, and a policy needs the row of that situation. Counting it
by hand is slow, so a function does it.

### 🎯 Task A. The row of a situation

Write `row_of(seats_left, day)`: go through `SITUATIONS` with a counter, and give back the row of the situation
whose seats left and day are these.

In [ ]:
# ✅ SOLUTION, task A: the row of a situation
def row_of(seats_left, day):
    # Gives the row of the table for a situation: the place of [seats_left, day] in SITUATIONS.
    row = 0                                     # the row of the situation we look at
    for situation in SITUATIONS:                # situation[0]: its seats left, situation[1]: its day
        if situation[0] == seats_left and situation[1] == day:     # the seats left and the day both match
            return row
        row = row + 1                           # the next situation has the next row

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# situation is a pair such as [4, 3]: situation[0] is its seats left, 4, and situation[1] its day, 3. The if
# asks whether both match, joined by and.
# When they do not match, the counter row moves on by 1.

In [ ]:
#@title 💡 Hint 2 for task A (double-click to open) { display-mode: "form" }
#         if situation[0] == seats_left and situation[1] == day:
#         row = row + 1

In [ ]:
# Check task A
check_row_of(row_of)

In [ ]:
# The row of DAY 3 with 2 seats left
found = play(row_of, 2, 3)
if found is not None:                           # once row_of works
    print("DAY 3 with 2 seats left is row", found)

## 4. A roulette

To follow a probability table, the plane needs a price sampled with the probabilities of its row. A roulette
does it. Cut a wheel of length 1 into two slots, one for each price, each as long as its probability. Row 0, 0.8 and
0.2, cuts it like this: PREMIUM from 0 to 0.8, STANDARD from 0.8 to 1.

Then throw a ball. It lands somewhere between 0 and 1, every place as likely as any other, and the slot it lands
in is the price. PREMIUM's slot is 0.8 long, so the ball lands in it in 80% of the throws.

In [ ]:
# Row 0's roulette: each price's slot, on a wheel from 0 to 1
show_roulette(my_table[0])

The kit's `spin()` throws the ball: it gives a number between 0 and 1, every number as likely as any other. The
next cell throws one ball on row 0's roulette and marks where it is. Run it a few times.

In [ ]:
# One ball on row 0's roulette
ball = spin()
show_roulette(my_table[0], ball)

### The roulette in Python, given

Two functions make a roulette and read it. They are given: run the next cell, and read them if you like.

- `make_roulette(probabilities)` gives the ends of the slots, a running total of the probabilities: each slot ends
  where the slot before it ended, plus its own probability. Row 0, 0.8 and 0.2, gives the ends 0.8 and 1.
- `where_it_lands(ends, ball)` gives the slot where a ball lands: the first slot whose end is above the ball. A
  slot is a number: 0 for PREMIUM's slot, 1 for STANDARD's.
- The kit's `spin()` throws the ball: a number between 0 and 1, every number as likely as any other.

In [ ]:
# Given: the ends of the slots of a roulette
def make_roulette(probabilities):
    # probabilities: a row of probabilities, one per price.
    # Gives the ends of the slots: a running total of the probabilities, such as [0.8, 1.0].
    ends = []
    running = 0                         # where the slots so far end: nowhere yet
    for probability in probabilities:
        running = running + probability  # this slot ends one probability further
        ends.append(running)             # remember where this slot ends
    return ends


# Given: the slot where the ball lands
def where_it_lands(ends, ball):
    # ends: the ends of the slots, from make_roulette. ball: a number between 0 and 1.
    # Gives the number of the first slot whose end is above the ball: 0 for PREMIUM, 1 for STANDARD.
    for slot in range(len(ends)):
        if ball < ends[slot]:           # the ball is below the end of this slot
            return slot
    return len(ends) - 1                # a ball at the very end of the wheel: the last slot


print("✅ make_roulette and where_it_lands are ready.")

In [ ]:
# Row 0's roulette, and the slot of a ball at 0.85
ends = make_roulette([0.8, 0.2])                  # row 0 of my_table
print("The ends of row 0's slots:", ends)
print("A ball at 0.85 lands in slot", where_it_lands(ends, 0.85))

### Where the ball lands

The ball lands in the first slot whose end is above it. With the ends 0.8 and 1, a ball at 0.85 is not below 0.8, so
it is not in PREMIUM's slot; it is below 1, so it lands in STANDARD's slot, slot 1.

💭 **Think first.** With the ends of row 2, 0.6 and 1, where does a ball at 0.35 land? And a ball at exactly 0.6?

In [ ]:
#@title The answer (double-click to open) { display-mode: "form" }
# A ball at 0.35 is below 0.6, so it lands in PREMIUM's slot.
# A ball at exactly 0.6 is not below 0.6, since it is equal; it is below 1, so it lands in STANDARD's slot. A
# slot holds its start and stops just before its end.

### Many throws

One throw can land anywhere. Many throws land in each slot about as often as its probability: the next cell throws
the ball 1,000 times on row 0's roulette, with make_roulette and where_it_lands. Run it a few times.

In [ ]:
# 1,000 throws on row 0's roulette
throw_balls(my_table[0], 1000)

**Reading the result.** The shares are close to the probabilities and never exactly equal: a roulette follows its
probabilities on average, and each set of throws has its own luck.

## 5. Picking a price

In a situation, the plane's price comes from the roulette of that situation's row in the probability table. A
probability table is a policy, so the function calls it `policy`.

### 🎯 Task B. A price at random

Write `pick_action(policy, row)`; on the plane an action is a price. It takes four steps, one line each, and
three of them call a function you met above:

1. The probabilities: the row of the situation, `policy[row]`.
2. The roulette: `make_roulette(probabilities)` gives the ends of its slots.
3. The ball: `spin()`, already in the skeleton.
4. The price: `where_it_lands(ends, ball)` gives the slot where the ball lands, and the slot is the price.

In [ ]:
# ✅ SOLUTION, task B: a price at random, with the probabilities of the situation's row
def pick_action(policy, row):
    # policy: a probability table, one row per situation. row: the row of today's situation.
    # Gives a price sampled with the probabilities of that row: 0 for PREMIUM, 1 for STANDARD.
    probabilities = policy[row]          # step 1: the row of probabilities
    ends = make_roulette(probabilities)  # step 2: its roulette
    ball = spin()                        # step 3: the ball
    return where_it_lands(ends, ball)    # step 4: where the ball lands

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The row of probabilities is policy[row]. make_roulette turns it into the ends of the slots.
# The ball is thrown for you. where_it_lands says where it lands, given the ends and the ball.

In [ ]:
#@title 💡 Hint 2 for task B (double-click to open) { display-mode: "form" }
#     probabilities = policy[row]
#     ends = make_roulette(probabilities)
#     return where_it_lands(ends, ball)

In [ ]:
# Check task B
check_pick_action(pick_action)

In [ ]:
# Your pick_action, 1,000 times in each row of my_table
show_picks(my_table, 1000)

**Reading the result.** Each row of shares is close to the same row of `my_table`: DAY 1 mostly PREMIUM, DAY 4
mostly STANDARD.

## 6. One day of the plane

The plane itself is in the kit: `sell(seats_left, price)`, the transition dynamics of the game. With the seats
left and a price, luck decides how many buyers come, with the probabilities of section 1, and `sell` gives back two
things: the seats sold and the day's money. Two names on the left of `=` catch them:

```python
seats_sold, money = sell(5, PREMIUM)
```

The money of a day is the seats sold times the price: \$5 a seat at PREMIUM, \$2 at STANDARD. Run the next cell a
few times: the same seats left and price do not always give the same day.

In [ ]:
# One day at PREMIUM with 5 seats left
seats_sold, money = sell(5, PREMIUM)
show_day(seats_sold, money)

### 🎯 Warm-up 3. One day at STANDARD

Call `sell` yourself, once: 5 seats left, and the price STANDARD.

In [ ]:
# ✅ SOLUTION, warm-up 3: one day at STANDARD with 5 seats left
seats_sold, money = sell(5, STANDARD)           # 5 seats left, then the price STANDARD
show_day(seats_sold, money)

In [ ]:
#@title 💡 Hint 1 for warm-up 3 (double-click to open) { display-mode: "form" }
# sell takes two inputs: first the seats left, then the price. The kit's name STANDARD stands for its number.

In [ ]:
#@title 💡 Hint 2 for warm-up 3 (double-click to open) { display-mode: "form" }
#     seats_sold, money = sell(5, STANDARD)

In [ ]:
# Check warm-up 3
check_day(seats_sold, money)

### Two days

A flight is a chain of such days. Each day, `row_of` gives the row of the situation, the policy picks a price in
that row, `sell` plays the day, and the seats sold are gone: the next day starts from the seats that are left.

### 🎯 Task C. Two days, written out

Write `two_days(policy)`: DAY 1 and DAY 2 of a flight, one after the other. DAY 1 is written for you; DAY 2 is
the same lines with the day 2. A flight never sells out on DAY 1, since STANDARD sells at most 3 seats.

In [ ]:
# ✅ SOLUTION, task C: the first two days of a flight, written out
def two_days(policy):
    # policy: a probability table. Plays DAY 1 and DAY 2 of a flight, one after the other.
    # Gives the money of the two days, in a list.
    seats_left = SEATS                              # nobody has bought a seat yet
    row = row_of(seats_left, 1)                     # DAY 1: the row of the situation
    price = pick_action(policy, row)                # DAY 1: the policy picks a price
    seats_sold, money_1 = sell(seats_left, price)   # DAY 1: the plane's day
    seats_left = seats_left - seats_sold            # DAY 1: the seats sold are gone
    row = row_of(seats_left, 2)                     # DAY 2: the row of the situation
    price = pick_action(policy, row)                # DAY 2: the policy's price
    seats_sold, money_2 = sell(seats_left, price)   # DAY 2: the plane's day
    return [money_1, money_2]

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# DAY 2 is DAY 1's first three lines again: the row of the situation, the policy's price, the plane's day.
# The row of DAY 2 comes from the seats that DAY 1 left, seats_left, and the day 2. The day's money is
# money_2.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
#     row = row_of(seats_left, 2)
#     price = pick_action(policy, row)
#     seats_sold, money_2 = sell(seats_left, price)

In [ ]:
# Check task C
check_two_days(two_days)

In [ ]:
# Two days of my_table's policy: run it a few times
show_money("The two days' money:", play(two_days, my_table))

## 7. A flight

A flight lasts until takeoff, after DAY 4, or until a FULL FLIGHT, whichever comes first. Its rewards are the
money of its days, in order. The next cell plays a flight with your `row_of` and `pick_action` and prints it day
by day. Run it a few times.

In [ ]:
# A flight of my_table's policy, day by day
show_flight(my_table)

### 🎯 Task D. A flight

Write `play_flight(policy)`: the lines of a day inside a loop over the days, each day's money appended to the
list of rewards, and the seats sold taken away. The `if` skips the days after a FULL FLIGHT.

In [ ]:
# ✅ SOLUTION, task D: a flight
def play_flight(policy):
    # policy: a probability table. Plays a flight from DAY 1 with 5 seats left, day after day.
    # Gives its rewards: the money of every day it sells on, in a list.
    rewards = []
    seats_left = SEATS
    for day in range(1, DAYS + 1):                  # DAY 1, 2, 3 and 4
        if seats_left > 0:                          # a FULL FLIGHT sells nothing more
            row = row_of(seats_left, day)           # the row of today's situation
            price = pick_action(policy, row)        # the policy picks a price
            seats_sold, money = sell(seats_left, price)     # the plane's day
            rewards.append(money)                   # remember the day's money
            seats_left = seats_left - seats_sold    # the seats sold today are gone
    return rewards

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# The body of the if is one day of task C, with the loop's day: the row from row_of(seats_left, day), the
# price from pick_action, the day from sell.
# rewards.append(...) keeps the day's money, and seats_left goes down by the seats sold.

In [ ]:
#@title 💡 Hint 2 for task D (double-click to open) { display-mode: "form" }
#     row = row_of(seats_left, day)
#     price = pick_action(policy, row)
#     seats_sold, money = sell(seats_left, price)
#     rewards.append(money)
#     seats_left = seats_left - seats_sold

In [ ]:
# Check task D
check_play_flight(play_flight)

### The return

On page 2 of Old Bessie, a later week counted a little less. A flight lasts at most 4 days, and the game adds its
days plainly: with $r_1, r_2, r_3, r_4$ the money of the days, the flight's MONEY, its return, is

$$G_1 = r_1 + r_2 + r_3 + r_4$$

### 🎯 Task E. The return of a flight

Write `episode_return(rewards)`: walk along the rewards and add each day's money to a total.

In [ ]:
# ✅ SOLUTION, task E: the return of a flight, its MONEY
def episode_return(rewards):
    # rewards: the money of every day of a flight, in a list.
    # Gives the flight's return: the plain sum of its days, no discount.
    total = 0
    for money in rewards:
        total = total + money           # add the day's money
    return total                        # the total

In [ ]:
#@title 💡 Hint 1 for task E (double-click to open) { display-mode: "form" }
# The total grows by the day's money in every round of the loop. After the loop, give the total back.

In [ ]:
#@title 💡 Hint 2 for task E (double-click to open) { display-mode: "form" }
#     total = total + money
#     return total

In [ ]:
# Check task E
check_episode_return(episode_return)

In [ ]:
# A flight of my_table's policy and its return: run it a few times
rewards = play(play_flight, my_table)
show_money("The flight's rewards:", rewards)
show_money("Its MONEY, the return:", play(episode_return, rewards))

## What you built

- **The situations**: `row_of` gives the row of a situation, `[seats left, day]`.
- **The roulette**: `pick_action` spins the roulette of a row with the given `make_roulette` and
  `where_it_lands`, and gives the price.
- **A flight**: `two_days` and `play_flight` play the plane day after day, and `episode_return` gives a
  flight's return.

Next, part 2 builds the forward pass: many flights, their average, and the average return of a table of preferences:
[proto-REINFORCE, Last-Minute Pricing, part 2: the forward pass](https://colab.research.google.com/github/eth-ainit-hs26/w3-cx-public/blob/main/03_cx_proto_reinforce_2b_last_minute_pricing_forward_pass_solutions.ipynb).